# Analisis Univariate - Dataset Penjualan Rumah
**Hari Selasa - Minggu 6 | Simulasi Industri Junior Data Analyst**

Notebook ini berisi analisis univariate (satu variabel) terhadap dataset `dataset_PenjualanRumah.xlsx`, mencakup histogram, bar chart, pie chart, dan boxplot beserta interpretasinya.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_excel('dataset_PenjualanRumah.xlsx')
df.head()

## 1. Gambaran Umum Dataset

Dataset berisi **1.010 data rumah** dijual dengan kolom:
- `NAMA RUMAH`: judul listing
- `HARGA`: harga jual (Rp)
- `LB`: luas bangunan (m2)
- `LT`: luas tanah (m2)
- `KT`: jumlah kamar tidur
- `KM`: jumlah kamar mandi
- `GRS`: kapasitas garasi

Tidak ada missing value pada dataset ini.

In [ ]:
print('Ukuran dataset:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nStatistik deskriptif:')
df.describe()

## Visualisasi 1: Histogram - Distribusi Harga Rumah

In [ ]:
def rupiah_fmt(x, pos):
    if x >= 1e9:
        return f'{x/1e9:.0f} M'
    return f'{x/1e6:.0f} jt'

fig, ax = plt.subplots(figsize=(8,5))
ax.hist(df['HARGA'], bins=30, color='#2E6F9E', edgecolor='white')
ax.set_title('Distribusi Harga Rumah', fontsize=14, fontweight='bold')
ax.set_xlabel('Harga (Rp)')
ax.set_ylabel('Jumlah Rumah')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(rupiah_fmt))
plt.tight_layout()
plt.show()

**Interpretasi:** Distribusi harga rumah menceng ke kanan (*right-skewed*). Median harga (~Rp5 miliar) jauh lebih rendah dari mean (~Rp7,6 miliar), menandakan sebagian kecil rumah bernilai sangat tinggi (hingga Rp65 miliar) menarik rata-rata ke atas. Mayoritas rumah berada di rentang harga Rp3-9 miliar.

## Visualisasi 2: Histogram - Distribusi Luas Tanah (LT)

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(df['LT'], bins=30, color='#4FA37B', edgecolor='white')
ax.set_title('Distribusi Luas Tanah (LT)', fontsize=14, fontweight='bold')
ax.set_xlabel('Luas Tanah (m2)')
ax.set_ylabel('Jumlah Rumah')
plt.tight_layout()
plt.show()

**Interpretasi:** Luas tanah juga menceng ke kanan. Sebagian besar rumah memiliki luas tanah 90-290 m2 (Q1-Q3), tetapi ada ekor panjang hingga 1.400 m2 yang menunjukkan segmen rumah mewah/tanah luas dalam jumlah kecil.

## Visualisasi 3: Bar Chart - Distribusi Jumlah Kamar Tidur (KT)

In [ ]:
kt_counts = df['KT'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(kt_counts.index.astype(str), kt_counts.values, color='#E8834B')
ax.set_title('Distribusi Jumlah Kamar Tidur (KT)', fontsize=14, fontweight='bold')
ax.set_xlabel('Jumlah Kamar Tidur')
ax.set_ylabel('Jumlah Rumah')
plt.tight_layout()
plt.show()

**Interpretasi:** Rumah dengan 4 dan 5 kamar tidur paling umum dijual (masing-masing 295 dan 270 unit), mencerminkan target pasar rumah keluarga menengah-besar. Rumah dengan >8 kamar tidur relatif jarang (segmen niche/rumah kos atau mewah).

## Visualisasi 4: Pie Chart - Proporsi Kapasitas Garasi (GRS)

In [ ]:
def bin_grs(x):
    if x == 0: return '0 (Tanpa Garasi)'
    elif x == 1: return '1'
    elif x == 2: return '2'
    else: return '3+'

df['GRS_BIN'] = df['GRS'].apply(bin_grs)
grs_counts = df['GRS_BIN'].value_counts().reindex(['0 (Tanpa Garasi)','1','2','3+'])

colors = ['#2E6F9E','#E8834B','#4FA37B','#C9564A']
fig, ax = plt.subplots(figsize=(7,7))
ax.pie(grs_counts.values, labels=grs_counts.index, autopct='%1.1f%%',
       colors=colors, startangle=90, wedgeprops={'edgecolor':'white','linewidth':1.5})
ax.set_title('Proporsi Kapasitas Garasi (GRS)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretasi:** Mayoritas rumah (35,6%) memiliki garasi berkapasitas 2 mobil, diikuti kapasitas 1 mobil (28,7%). Sekitar 13% rumah tidak memiliki garasi sama sekali, sementara sisanya (>22%) memiliki garasi luas (3 mobil atau lebih).

## Visualisasi 5: Boxplot - Deteksi Outlier Harga Rumah

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.boxplot(df['HARGA'], vert=False, patch_artist=True,
           boxprops=dict(facecolor='#2E6F9E', alpha=0.7),
           medianprops=dict(color='#E8834B', linewidth=2))
ax.set_title('Boxplot Harga Rumah (Deteksi Outlier)', fontsize=14, fontweight='bold')
ax.set_xlabel('Harga (Rp)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(rupiah_fmt))
ax.set_yticks([])
plt.tight_layout()
plt.show()

Q1, Q3 = df['HARGA'].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper = Q3 + 1.5*IQR
outliers = df[df['HARGA'] > upper]
print(f'Batas atas outlier (IQR method): Rp{upper:,.0f}')
print(f'Jumlah rumah outlier (harga tinggi): {len(outliers)} dari {len(df)} ({len(outliers)/len(df)*100:.1f}%)')

**Interpretasi:** Boxplot mengonfirmasi kemencengan distribusi harga. Terdapat sekitar 94 rumah (9,3%) yang tergolong outlier harga tinggi berdasarkan metode IQR, dengan whisker atas berhenti jauh di bawah harga maksimum (Rp65 miliar) — menandakan segmen kecil rumah super-mewah yang berbeda karakteristik pasarnya dari mayoritas listing.

## Ringkasan Statistik Deskriptif Lengkap

In [ ]:
cols = ['HARGA','LB','LT','KT','KM','GRS']
summary = pd.DataFrame({
    'Mean': df[cols].mean(),
    'Median': df[cols].median(),
    'Modus': df[cols].mode().iloc[0],
    'Std Dev': df[cols].std(),
    'Min': df[cols].min(),
    'Max': df[cols].max(),
    'Q1': df[cols].quantile(0.25),
    'Q3': df[cols].quantile(0.75),
})
summary

## Insight Utama

1. **Harga rumah sangat bervariasi dan menceng kanan** — sebagian kecil listing rumah mewah (>Rp20 M) menarik rata-rata jauh di atas median, sehingga median lebih representatif sebagai patokan harga pasar.
2. **Segmen rumah keluarga (4-5 KT) mendominasi pasar**, mengindikasikan permintaan/penawaran terbesar ada di kelas menengah-atas keluarga besar.
3. **Garasi 2 mobil adalah standar pasar** — cocok jadi fitur "default" saat membandingkan listing.
4. **~9% listing merupakan outlier harga tinggi**, layak dianalisis terpisah agar tidak mendistorsi model harga rata-rata (misalnya untuk pricing model, sebaiknya di-cap atau ditangani khusus).
5. **Luas tanah dan luas bangunan sama-sama menceng kanan**, konsisten dengan pola harga — mengindikasikan luas properti kemungkinan menjadi salah satu pendorong utama harga (akan dikonfirmasi di analisis bivariate/korelasi).

---
# Analisis Korelasi Data
**Hari Kamis - Minggu 6**

Bagian ini membuat correlation matrix, heatmap, dan interpretasi korelasi antarvariabel numerik pada dataset, untuk mengidentifikasi variabel yang paling berpengaruh terhadap harga rumah.

In [ ]:
cols = ['HARGA','LB','LT','KT','KM','GRS']
corr = df[cols].corr(method='pearson')
corr.round(3)

## Heatmap Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8,7))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha='right')
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        val = corr.iloc[i,j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=10, fontweight='bold')
ax.set_title('Correlation Matrix Heatmap - Dataset Penjualan Rumah', fontsize=13, fontweight='bold', pad=15)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Koefisien Korelasi (Pearson)')
plt.tight_layout()
plt.show()

## Variabel Paling Berpengaruh terhadap HARGA

In [ ]:
corr_harga = corr['HARGA'].drop('HARGA').sort_values(ascending=False)
corr_harga

**Interpretasi Korelasi:**

- **LT (Luas Tanah)** memiliki korelasi terkuat dengan HARGA (r = 0,81, kategori *kuat*) — variabel **independen** paling berpengaruh terhadap harga.
- **LB (Luas Bangunan)** juga berkorelasi kuat dengan HARGA (r = 0,75).
- **GRS, KM, KT** berkorelasi moderat-lemah dengan HARGA (r = 0,32-0,48).
- **KT dan KM** berkorelasi cukup kuat satu sama lain (r = 0,67) — wajar karena rumah dengan lebih banyak kamar tidur umumnya juga punya lebih banyak kamar mandi (potensi *multikolinearitas* bila dipakai bersama dalam model regresi).
- **HARGA** berperan sebagai variabel **dependen** (yang ingin diprediksi/dijelaskan) dalam analisis ini.

**Ringkasan Hasil:** Luas properti (LT dan LB) adalah pendorong utama harga rumah dalam dataset ini, jauh lebih dominan dibanding jumlah kamar tidur/mandi atau kapasitas garasi. Untuk pemodelan harga (misalnya regresi), LT dan LB layak menjadi fitur utama, sementara KT dan KM sebaiknya tidak digunakan bersamaan tanpa penanganan multikolinearitas.